# Analyse du Réseau des Philosophes : Détection de Communautés et Modularité

Ce notebook contient l'implémentation complète, modulaire et documentée des 5 tâches d'analyse de graphes sur le réseau des philosophes de Wikipédia.

### Sommaire :
0. **Pré-traitement** : Chargement, agrégation des poids symétriques et extraction de la composante géante (GCC).
1. **Tâche 1** : Exécution de Louvain (`networkx.community.louvain_communities`) et calcul de la modularité $Q$.
2. **Tâche 2** : Top 5 des philosophes par degré dans chaque communauté et score NMI avec `era` et `subfields`.
3. **Tâche 3** : Robustesse stochastique (5 seeds), matrice NMI 5x5 et identification des philosophes instables.
4. **Tâche 4** : Clauset-Newman-Moore (`greedy_modularity_communities`) vs Louvain.
5. **Tâche 5** : Implémentation manuelle *from scratch* de la Phase 1 de Louvain et validation sur le *Karate Club* de Zachary.



In [ ]:
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics import normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import seaborn as sns

print("Bibliothèques importées avec succès !")



## 0. Chargement et extraction de la composante géante (GCC)
- On combine les liens orientés $(u \to v)$ et $(v \to u)$ en sommant leurs poids pour obtenir un réseau non orienté pondéré.
- On extrait ensuite la plus grande composante connexe (GCC).



In [ ]:
# 1. Chargement des fichiers TSV
df_nodes = pd.read_csv('week4_philosophers_nodes.tsv', sep='\t', comment='#')
df_edges = pd.read_csv('week4_philosophers_edges.tsv', sep='\t', comment='#')

# 2. Construction du graphe non orienté
G = nx.Graph()

for _, row in df_nodes.iterrows():
    G.add_node(row['node_id'], **row.to_dict())

for _, row in df_edges.iterrows():
    u, v, w = row['source'], row['target'], float(row['weight'])
    if G.has_edge(u, v):
        G[u][v]['weight'] += w
    else:
        G.add_edge(u, v, weight=w)

# 3. Extraction de la composante géante (GCC)
gcc_nodes = max(nx.connected_components(G), key=len)
gcc = G.subgraph(gcc_nodes).copy()

print(f"Graphe global : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes")
print(f"Composante géante (GCC) : {gcc.number_of_nodes()} nœuds, {gcc.number_of_edges()} arêtes")
print(f"Poids total dans la GCC : {sum(d['weight'] for _, _, d in gcc.edges(data=True)):.1f}")



## Tâche 1 : Algorithme de Louvain (`louvain_communities`) et modularité $Q$


In [ ]:
seed_task1 = 42
comms_louvain_1 = nx.community.louvain_communities(gcc, weight='weight', seed=seed_task1)
q_louvain_1 = nx.community.modularity(gcc, comms_louvain_1, weight='weight')

print("=" * 60)
print("TÂCHE 1 : RÉSULTATS LOUVAIN")
print("=" * 60)
print(f"Nombre de communautés détectées : {len(comms_louvain_1)}")
print(f"Modularité Q                    : {q_louvain_1:.4f}")
print(f"Tailles des communautés         : {sorted([len(c) for c in comms_louvain_1], reverse=True)}")



## Tâche 2 : Top 5 membres par degré et calcul NMI avec `era` et `subfields`


In [ ]:
# 1. Affichage des 5 membres les plus connectés pour chaque communauté
sorted_comms = sorted(comms_louvain_1, key=len, reverse=True)

print("=" * 60)
print("TOP 5 PHILOSOPHES PAR DEGRÉ DANS CHAQUE COMMUNAUTÉ")
print("=" * 60)
for idx, comm in enumerate(sorted_comms, 1):
    top5 = sorted(comm, key=lambda n: gcc.degree(n), reverse=True)[:5]
    top5_str = ", ".join([f"{n} (k={gcc.degree(n)})" for n in top5])
    print(f"Communauté {idx:02d} ({len(comm):3d} nœuds) : {top5_str}")

# 2. Préparation des labels
node_order = sorted(list(gcc.nodes()))
node2comm = {node: cid for cid, c in enumerate(sorted_comms) for node in c}
labels_comm = [node2comm[n] for n in node_order]

def get_clean_attr(n, attr):
    val = gcc.nodes[n].get(attr, 'none')
    if pd.isna(val) or str(val).strip() == '':
        return 'none'
    return str(val).strip()

labels_era = [get_clean_attr(n, 'era') for n in node_order]
labels_subfields = [get_clean_attr(n, 'subfields') for n in node_order]

# 3. Calcul de la NMI (Normalized Mutual Information)
nmi_era = normalized_mutual_info_score(labels_era, labels_comm)
nmi_subfields = normalized_mutual_info_score(labels_subfields, labels_comm)

print("\n" + "=" * 60)
print("SCORES NORMALIZED MUTUAL INFORMATION (NMI)")
print("=" * 60)
print(f"NMI(Louvain, era)       : {nmi_era:.4f}")
print(f"NMI(Louvain, subfields) : {nmi_subfields:.4f}")



## Tâche 3 : Robustesse sur 5 seeds, Matrice NMI 5x5 et Philosophes Instables


In [ ]:
seeds = [1, 42, 123, 456, 789]
history = []
partitions_labels = []

for s in seeds:
    c = nx.community.louvain_communities(gcc, weight='weight', seed=s)
    q = nx.community.modularity(gcc, c, weight='weight')
    history.append({'seed': s, 'nb_communautes': len(c), 'modularite_Q': q})
    n2c = {node: cid for cid, comm in enumerate(c) for node in comm}
    partitions_labels.append([n2c[n] for n in node_order])

df_seeds = pd.DataFrame(history)
print("Statistiques par seed :")
print(df_seeds.to_string(index=False))

# Matrice NMI 5x5
n_seeds = len(seeds)
nmi_matrix = np.zeros((n_seeds, n_seeds))
for i in range(n_seeds):
    for j in range(n_seeds):
        nmi_matrix[i, j] = normalized_mutual_info_score(partitions_labels[i], partitions_labels[j])

df_nmi = pd.DataFrame(nmi_matrix, 
                      index=[f'Seed {s}' for s in seeds], 
                      columns=[f'Seed {s}' for s in seeds])

print("\nMatrice NMI 5x5 :")
print(df_nmi.round(4))

# Heatmap visuelle
plt.figure(figsize=(6, 5))
sns.heatmap(df_nmi, annot=True, cmap='Blues', fmt='.3f', vmin=0.6, vmax=1.0)
plt.title('Similarité NMI entre 5 seeds de Louvain')
plt.tight_layout()
plt.show()

# Mesure d'instabilité des nœuds
# A. Consensus de co-appartenance
n_nodes = len(node_order)
co_mat = np.zeros((n_nodes, n_nodes), dtype=float)
for lab in partitions_labels:
    arr = np.array(lab)
    co_mat += (arr[:, None] == arr[None, :])
co_mat /= n_seeds

instability = np.sum(4 * co_mat * (1.0 - co_mat), axis=1) / (n_nodes - 1)

# B. Alignement hongrois des labels
aligned_labels = [partitions_labels[0]]
for i in range(1, n_seeds):
    ref = partitions_labels[0]
    curr = partitions_labels[i]
    cost = np.zeros((max(curr) + 1, max(ref) + 1), dtype=int)
    for c, r in zip(curr, ref):
        cost[c, r] -= 1
    r_ind, c_ind = linear_sum_assignment(cost)
    mapping = {r: c for r, c in zip(r_ind, c_ind)}
    aligned_labels.append([mapping.get(c, c) for c in curr])

aligned_arr = np.array(aligned_labels).T
n_unique_comms = [len(np.unique(row)) for row in aligned_arr]

df_instables = pd.DataFrame({
    'philosophe': node_order,
    'degree': [gcc.degree(n) for n in node_order],
    'score_instabilite': instability,
    'nb_communautes_distinctes': n_unique_comms,
    'era': [gcc.nodes[n].get('era', 'none') for n in node_order]
}).sort_values(by=['score_instabilite', 'nb_communautes_distinctes'], ascending=[False, False])

print("\nTop 10 des philosophes changeant le plus souvent de communauté :")
print(df_instables.head(10).to_string(index=False))



## Tâche 4 : Algorithme Glouton (`greedy_modularity_communities`) vs Louvain


In [ ]:
greedy_comms = nx.community.greedy_modularity_communities(gcc, weight='weight')
q_greedy = nx.community.modularity(gcc, greedy_comms, weight='weight')

n2c_greedy = {node: cid for cid, c in enumerate(greedy_comms) for node in c}
labels_greedy = [n2c_greedy[n] for n in node_order]

nmi_greedy_louvain = normalized_mutual_info_score(labels_comm, labels_greedy)

print("=" * 60)
print("COMPARAISON GREEDY (Clauset-Newman-Moore) VS LOUVAIN")
print("=" * 60)
print(f"Greedy Modularity - Nombre de communautés : {len(greedy_comms)}")
print(f"Greedy Modularity - Modularité Q          : {q_greedy:.4f}")
print(f"Louvain (Seed 42) - Nombre de communautés : {len(comms_louvain_1)}")
print(f"Louvain (Seed 42) - Modularité Q          : {q_louvain_1:.4f}")
print(f"NMI(Greedy, Louvain Seed 42)              : {nmi_greedy_louvain:.4f}")



## Tâche 5 : Implémentation manuelle de la Phase 1 de Louvain (*from scratch*)

### Formulation mathématique du gain $\Delta Q$ :
Pour un nœud $u$ retiré de sa communauté d'origine et inséré dans une communauté candidate voisine $C$ :

$$\Delta Q(u \to C) = \frac{k_{u, in}^C}{m} - \frac{\Sigma_{tot}^C \cdot k_u}{2 m^2}$$

où :
- $m = \sum_{e \in E} w(e)$ est la somme des poids des arêtes du réseau,
- $k_u = \sum_{v} w(u, v)$ est le degré pondéré de $u$,
- $\Sigma_{tot}^C = \sum_{v \in C, v \neq u} k_v$ est la somme des degrés pondérés des membres de $C$,
- $k_{u, in}^C = \sum_{v \in C, v \neq u} w(u, v)$ est la somme des poids des arêtes reliant $u$ aux membres de $C$.



In [ ]:
def compute_modularity_manual(G, communities, weight='weight'):
    """Calcul from scratch de la modularité Q de Newman."""
    m = G.size(weight=weight)
    if m == 0:
        return 0.0
    two_m = 2.0 * m
    k = dict(G.degree(weight=weight))
    Q = 0.0
    for comm in communities:
        subG = G.subgraph(comm)
        w_in = subG.size(weight=weight)
        w_tot = sum(k[n] for n in comm)
        Q += (2.0 * w_in / two_m) - (w_tot / two_m) ** 2
    return Q


def louvain_phase_1(G, weight='weight', seed=None, max_passes=50):
    """Implémentation from scratch de la Phase 1 de Louvain."""
    if G.is_directed():
        raise ValueError("Le graphe doit être non orienté.")
    if seed is not None:
        random.seed(seed)
        
    nodes = list(G.nodes())
    m = G.size(weight=weight)
    if m == 0:
        return [{n} for n in nodes], 0
        
    k = dict(G.degree(weight=weight))
    
    # Initialisation : chaque nœud dans sa communauté singleton
    community = {n: i for i, n in enumerate(nodes)}
    sigma_tot = {i: float(k[n]) for i, n in enumerate(nodes)}
    
    passes = 0
    next_singleton_id = len(nodes)
    
    while passes < max_passes:
        passes += 1
        moved = False
        order = list(nodes)
        if seed is not None:
            random.shuffle(order)
            
        for u in order:
            c_old = community[u]
            k_u = k[u]
            
            # Retrait virtuel de u
            sigma_tot[c_old] -= k_u
            
            # Liens vers les communautés voisines
            neigh_weights = defaultdict(float)
            for v, data in G[u].items():
                if v == u:
                    continue
                neigh_weights[community[v]] += data.get(weight, 1.0)
                
            # Recherche du meilleur gain Delta Q
            best_comm = c_old
            best_gain = 0.0  # baseline : rester isolé
            
            for c, k_u_in in neigh_weights.items():
                gain = (k_u_in / m) - (sigma_tot[c] * k_u) / (2.0 * m * m)
                if gain > best_gain:
                    best_gain = gain
                    best_comm = c
                    
            if best_gain <= 0.0 and sigma_tot[c_old] > 0.0:
                best_comm = next_singleton_id
                next_singleton_id += 1
                best_gain = 0.0
                
            community[u] = best_comm
            sigma_tot[best_comm] = sigma_tot.get(best_comm, 0.0) + k_u
            
            if best_comm != c_old:
                moved = True
                
        if not moved:
            break
            
    grouped = defaultdict(set)
    for n, c in community.items():
        grouped[c].add(n)
    return list(grouped.values()), passes

# Validation sur Zachary's Karate Club
G_karate = nx.karate_club_graph()

manual_comms, passes = louvain_phase_1(G_karate, seed=42)
q_manual = compute_modularity_manual(G_karate, manual_comms)

nx_comms = nx.community.louvain_communities(G_karate, seed=42)
q_nx = nx.community.modularity(G_karate, nx_comms)

print("=" * 60)
print("BENCHMARK SUR ZACHARY'S KARATE CLUB")
print("=" * 60)
print(f"Phase 1 Manuelle : {len(manual_comms)} communautés en {passes} passes | Q = {q_manual:.4f}")
print(f"Louvain NetworkX : {len(nx_comms)} communautés              | Q = {q_nx:.4f}")
print(f"Tailles Phase 1 Manuelle : {[len(c) for c in manual_comms]}")
print(f"Tailles Louvain NetworkX : {[len(c) for c in nx_comms]}")

